# 基于多智能体与自进化的旅游路书生成系统

**架构概览**：简报官(Briefing) 把用户出行需求整理为出行简报并起草路书 → 协调官(Coordinator) 调度多个情报员(Scout)并行搜集景点/交通/住宿/美食情报 → 精修路书 → 评估器(Evaluator) 进行「可执行性/预算合理性/体验丰富度」三维自进化评分 → 体验官(Critic) 对抗挑刺 → 动态注入反馈迭代修复 → 撰写最终路书。

In [1]:
from importlib.metadata import version
print("langchain version: ", version("langchain"))
print("langgraph version: ", version("langgraph"))
print("langchain-openai version: ", version("langchain-openai"))
print("langchain-core version: ", version("langchain-core"))
print("langchain_community version: ", version("langchain_community"))
print("tavily-python version: ", version("tavily-python"))

langchain version:  1.3.11
langgraph version:  1.2.6
langchain-openai version:  1.3.3
langchain-core version:  1.4.8
langchain_community version:  0.4.2
tavily-python version:  0.7.26


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
required_keys = ["OPENAI_API_KEY", "TAVILY_API_KEY", "AMAP_MAPS_API_KEY"]
missing_keys = [name for name in required_keys if not os.getenv(name)]
if missing_keys:
    raise RuntimeError(f"缺少环境变量: {', '.join(missing_keys)}")
print("服务密钥检查通过（未显示密钥内容）")

服务密钥检查通过（未显示密钥内容）


In [3]:
from IPython.display import Markdown, display
from langgraph.checkpoint.memory import InMemorySaver
from travel_planner.itinerary_builder import itinerary_builder
from importlib import import_module
import warnings

warnings.filterwarnings("ignore")
coordinator_runtime = import_module("travel_planner.agents.coordinator")
coordinator_runtime.max_planning_iterations = 4
coordinator_runtime.max_concurrent_scouts = 2
checkpointer = InMemorySaver()
full_agent = itinerary_builder.compile(checkpointer=checkpointer)
graph = full_agent.get_graph(xray=True)
print("主图节点:", list(full_agent.get_graph().nodes))
print("Notebook 演示上限: 4 轮 Coordinator / 2 个并发 Scout")
display(Markdown(f"```mermaid\n{graph.draw_mermaid()}\n```"))

2026-08-14 14:20:25 [INFO] travel_planner.llm: Selected cognition backend 'openai' for role 'briefing' with handle 'glm-5.1' (timeout=600)


2026-08-14 14:20:27 [INFO] travel_planner.llm: Selected cognition backend 'openai' for role 'scout_summarizer' with handle 'glm-5.1' (timeout=600)


2026-08-14 14:20:27 [INFO] travel_planner.llm: Selected cognition backend 'openai' for role 'writer' with handle 'glm-5.1' (timeout=600)


2026-08-14 14:20:28 [INFO] travel_planner.llm: Selected cognition backend 'openai' for role 'scout_main' with handle 'glm-5.1' (timeout=600)


2026-08-14 14:20:28 [INFO] travel_planner.llm: Selected cognition backend 'openai' for role 'scout_compressor' with handle 'glm-5.1' (timeout=600)


...

2026-08-14 14:20:28 [INFO] travel_planner.llm: Selected cognition backend 'openai' for role 'evaluator' with handle 'glm-5.1' (timeout=600)


2026-08-14 14:20:28 [INFO] travel_planner.llm: Selected cognition backend 'openai' for role 'coordinator' with handle 'glm-5.1' (timeout=600)


2026-08-14 14:20:28 [INFO] travel_planner.llm: Selected cognition backend 'openai' for role 'writer' with handle 'glm-5.1' (timeout=600)


主图节点: ['__start__', 'plan_trip_brief', 'write_draft_itinerary', 'coordinator_subgraph', 'final_itinerary_generation', '__end__']
Notebook 演示上限: 4 轮 Coordinator / 2 个并发 Scout


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan_trip_brief(plan_trip_brief)
	write_draft_itinerary(write_draft_itinerary)
	coordinator_subgraph(coordinator_subgraph)
	final_itinerary_generation(final_itinerary_generation)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan_trip_brief;
	coordinator_subgraph --> final_itinerary_generation;
	plan_trip_brief --> write_draft_itinerary;
	write_draft_itinerary --> coordinator_subgraph;
	final_itinerary_generation --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [4]:
from pathlib import Path
from langchain_core.messages import HumanMessage

thread = {"configurable": {"thread_id": "run-notebook-1"}, "recursion_limit": 100}
request = {
    "user_id": "notebook-demo-user",
    "session_metadata": {
        "session_id": "run-notebook-1",
        "channel": "jupyter",
        "temp_flags": {"language": "zh-CN"},
        "requested_skills": [],
    },
    "profile_updates": {
        "party": "2大1小",
        "pace": "轻松",
        "interests": ["文化", "美食"],
    },
    "messages": [HumanMessage(content="我准备和家人（2大1小）去西安玩3天2夜，预算1.5万元，希望节奏轻松、亲子友好，重点体验历史文化与本地美食。请规划详细路书，包含景点、交通、住宿、美食、门票、预算和注意事项。")],
}
cached_result = Path("results/output_itinerary_sample_1.md")
if os.getenv("NOTEBOOK_REUSE_RESULT") == "1" and cached_result.exists():
    result = {"final_itinerary": cached_result.read_text(encoding="utf-8")}
    print("复用上一轮已完成的主图结果，仅续跑后续单元")
else:
    result = await full_agent.ainvoke(request, config=thread)
assert result.get("final_itinerary"), "主图没有生成最终路书"
print("主图执行完成，最终路书字符数:", len(result["final_itinerary"]))

复用上一轮已完成的主图结果，仅续跑后续单元
主图执行完成，最终路书字符数: 8214


In [5]:
from IPython.display import Markdown, display
display(Markdown(result["final_itinerary"]))

# 西安3天2夜家庭亲子文化美食之旅 · 深度执行路书

欢迎来到十三朝古都西安。本路书专为2大1小的家庭量身定制，以“历史启蒙与美食探索”为核心，全程保持轻松、亲子友好的节奏。行程总预算严格控制在1.5万元人民币以内，并为您预留了充足的弹性空间。

---

## 一、 行程总览与规划思路

本次3天2夜的行程设计遵循“由近及远、先缓后急”的原则。为了最大化亲子游览的舒适度，我们将市区内的轻松步行与外围的重头戏合理分布：第一天以古城墙骑行和回民街美食破冰，让孩子迅速融入西安的市井烟火；第二天穿越至盛唐，深度造访陕西历史博物馆与大唐不夜城；第三天则前往世界第八大奇迹——兵马俑，并在品尝非遗美食后圆满返程。

美食探索被无缝融入每日行程中，我们将避开过度商业化的主干道，带您寻找最地道的肉夹馍、羊肉泡馍与特色陕菜。考虑到您未提供具体的出行日期、孩子年龄及出发城市，本方案在交通、门票政策及游览节奏上均设置了灵活调整机制，确保您可以直接照此执行。

---

## 二、 开放性因素与弹性调整方案

由于部分出行细节尚未明确，本路书提供以下弹性指南，供您对照实际情况无缝调整：

首先是**大交通与出发地**。路书将以北京、上海、广州三大常见出发地为例提供参考。如果您选择自驾或乘坐高铁，可根据实际里程微调预算。
其次是**孩子的年龄跨度**。这将直接影响门票支出与体力分配：如果孩子在6岁以下，兵马俑和陕历博建议以“听故事”为主，避免长时间步行引起哭闹，城墙骑行建议乘坐电瓶车或租赁双人车；如果孩子在7-14岁，可以增加互动体验（如陶俑制作DIY），并在陕历博安排深度人工讲解；孩子满16周岁或身高超过1.5米后，部分景点的门票政策将有所变化，需按成人票标准购买。
再者是**具体出行日期**。西安四季分明，夏季（6-8月）炎热多雨，兵马俑坑内尤为闷热，需极度注意防暑；冬季（12-2月）干冷，城墙骑行需防寒。如果您在春节、国庆等法定节假日出行，所有热门景点必须提前7-10天掐点抢票。
最后是**饮食限制**。西安美食以面食、牛羊肉及辛辣调料（如油泼辣子）为主。如有花生、芝麻过敏或不吃牛羊肉，本路书在每日餐饮中均提供了备选的清淡粤菜或家常菜选项。

---

## 三、 大交通方案与抵达指引

针对未指定的出发城市，以下为您整理了从国内主要核心城市往返西安的交通参考方案。您可以根据自身所在城市，选择最合适的交通枢纽与出行方式。

| 出发城市 | 推荐交通方式 | 大致耗时 | 参考票价 (单程/成人) | 亲子出行贴士 |
| :--- | :--- | :--- | :--- | :--- |
| **北京** | 高铁优先 | 约4.5-5.5小时 | 二等座约515元 | 北京西直达西安北站，推荐购买3连座方便照顾孩子。 |
| **北京** | 飞机 | 约2小时 | 经济舱600-1200元 | 首都/大兴至咸阳机场，注意机场往返市区需额外耗时。 |
| **上海** | 高铁/飞机均可 | 约6-7小时/2.5小时 | 高铁二等座约669元 | 若孩子耐性足可体验高铁，否则建议飞往西安咸阳机场。 |
| **广州** | 飞机优先 | 约3小时 | 经济舱500-1500元 | 高铁需8-9小时，带儿童极易疲劳，强烈建议乘坐飞机。 |

*参考来源：中国铁路12306官网及各大航司直销平台 [1]。儿童票政策：高铁对满6周岁且未满14周岁的儿童可购买儿童优惠票；航空通常对满2周岁未满12周岁的儿童收取成人全价票的50%。*

**抵达后的市内接驳：**
无论您抵达**西安北站（高铁）**还是**西安咸阳国际机场**，接驳市中心都非常便捷。西安地铁网络已开通12条线路，票价亲民（短途2元起）。如果您抵达西安北站，可直接在站内换乘地铁2号线（蓝色），约40分钟即可直达市中心钟楼区域，单程票价约5元/人 [2]。如果您抵达咸阳机场，推荐乘坐地铁14号线，在西安北站内换乘2号线前往市区，全程约1小时；若行李较多，也可选择出租车或网约车，直达市中心钟楼区域约需100-120元，耗时40-60分钟 [3]。

---

## 四、 亲子住宿精选推荐

对于2大1小的家庭，住宿位置的选定直接决定了每天步行的疲劳度。推荐您将酒店定在**钟鼓楼/永宁门区域**或**大雁塔/曲江区域**。前者位于老城中心，楼下就是美食街与地铁2号线，生活气息浓厚；后者靠近曲江多个核心景区，环境更现代宽敞。

以下为您提供三个不同档次的选择，您可以根据1.5万元的充裕预算和个人喜好进行挑选：

| 酒店类型 | 推荐酒店名称 | 参考价位 | 推荐理由及亲子设施 |
| :--- | :--- | :--- | :--- |
| **舒适连锁型** | 西安钟楼希尔顿欢朋酒店 | 约500-700元/晚 | 位于钟楼商圈，步行可达回民街。标准统一，干净卫生，提供丰盛自助早餐，适合追求稳妥、性价比高的家庭。可申请安排家庭房或连通房。 |
| **精品文化型** | 西安城墙边精品民宿 (如：永宁门附近) | 约600-900元/晚 | 距离城墙南门极近，部分房型推窗可见城墙夜景。装修融入本地特色元素，更有在地文化体验感。建议提前致电确认是否提供儿童洗漱用品及加床服务。 |
| **豪华享受型** | 西安赛瑞喜来登大酒店 / 索菲特人民大厦 | 约800-1500元/晚 | 均位于市中心核心地段。拥有宽敞的室外恒温泳池（孩子放电好去处）和精美的园林景观，服务一流。含早日均价格较高，但能大幅提升度假的松弛感。 |

*(以上房价为平季参考，若逢暑期或春节等旺季，价格可能上浮30%-50%，建议提前1个月在携程等OTA平台锁定可免费取消的房源 [4]。)*

---

## 五、 每日详细行程规划

### Day 1：破冰之旅 —— 触摸古老城墙，品味烟火长安

**上午/中午：抵达西安，初识古都**
早上从出发地前往西安，中午时分办理入住手续并寄存行李。舟车劳顿后，第一顿餐食建议以轻便、出餐快为主。您可以在酒店附近寻找“魏家凉皮”（西安本土知名中式快餐连锁），点一份凉皮搭配柔软多汁的肉夹馍，人均仅需30元左右。如果孩子偏爱面食，可前往“子午路张记肉夹馍”，这里的优质腊汁肉夹馍肥而不腻，绝对能瞬间唤醒全家人的味蕾。

**下午：西安城墙（永宁门）亲子骑行**
午餐稍作休息后，下午2点左右前往西安明城墙。建议从**南门（永宁门）**登城，这里地铁2号线直达，且是规模最大的迎宾入口。城墙门票为成人54元/人，学生/1.2米以上儿童27元 [5]。
城墙全长13.74公里，对于孩子来说全程步行负担过重，**强烈推荐在城墙上租赁双人自行车**。租赁点位于南门东侧，双人车90元/3小时，押金100元 [5]。建议顺时针骑行至东门（长乐门），这一段约4公里，沿途既能俯瞰城内的青砖灰瓦，也能远眺城外的现代高楼。请注意，城墙地面为复古青砖，具有一定颠簸感，家长骑行需控制速度。3小时的租赁时间十分充裕，沿途可随时停下拍照休息。如果孩子太小无法骑行，南门也有全程80元/人的电瓶车观光车可供选择。

**晚间：钟鼓楼夜景与洒金桥美食夜市**
傍晚5点半左右下城墙，步行至钟鼓楼广场。此时华灯初上，钟楼和鼓楼在夜色中金碧辉煌，是绝佳的打卡拍照地（不建议登楼，外观更为壮观且省去奔波）。随后，避开人声鼎沸且商业化严重的回民街主街，带家人步行前往本地老饕聚集的**洒金桥**或**大皮院**。
在这里，您可以开展一场边走边吃的美食接力：先在“志亮灌汤蒸饺”品尝皮薄馅大的牛肉蒸饺，再去“花奶奶酸梅汤”喝一杯解腻的乌梅汁，最后来一份软糯香甜的“东南亚甑糕”（孩子一定会喜欢）。晚餐一家三口在小吃街的花费约在150元左右即可吃撑。

**Day 1 预算小结**
* 市内交通：约30元（地铁及短途打车）
* 景点门票：城墙135元（2大1小） + 自行车90元
* 餐饮：午餐90元 + 晚餐（洒金桥）150元
* **当日合计：约405元**

---

### Day 2：梦回大唐 —— 穿越历史长河，沉醉不夜之城

**上午：陕西历史博物馆（深度文化洗礼）**
今天的文化重头戏是被誉为“华夏宝库”的陕西历史博物馆。博物馆实行“免费不免票”政策，但**预约难度极高**。您必须提前在微信公众号“陕西历史博物馆”抢票，提前5天的下午17:00准时放票 [6]。如果实在抢不到免费票，您可以花30元购买“珍宝馆（何家村窖藏）”门票作为替代入馆方案 [7]。该馆周一闭馆，请务必避开。
为了让孩子不觉得枯燥，**非常建议您在飞猪或携程提前预订“门票+人工讲解”的半日游产品**（约150-200元/人）。专业的讲解员会用生动的故事讲述镶金兽首玛瑙杯背后的丝绸之路，以及兵马俑前世的传奇，让枯燥的文物在孩子眼中活起来。

**中午：长安大牌档（赛格店）**
中午从博物馆出来，步行可达小寨赛格国际购物中心。直奔六楼的“长安大牌档”，这里是沉浸式体验陕菜的绝佳之地，装修还原了唐代集市风貌。必点的“葫芦鸡”外酥里嫩，用手撕着吃极具满足感；造型别致的“毛笔酥”不仅拍照出片，口感也酥脆香甜，非常适合小朋友。人均消费约80-100元，由于生意火爆，建议提前在大众点评线上排号 [4]。

**下午：大雁塔与大唐芙蓉园**
下午2点半，前往大雁塔（大慈恩寺）。这里曾是唐僧（玄奘）翻译佛经的地方，您可以给孩子讲讲《西游记》的真实历史背景。大慈恩寺门票40元，登塔另需25元（儿童半价） [8]。
游览完毕后，步行约15分钟可达**大唐芙蓉园**。请注意，这里并非免费景点，成人门票为120元（提前一天购票或有特惠68元早场票，1.2米以下儿童及65岁以上老人免费） [9]。园内是全方位展示盛唐风貌的大型皇家园林，傍晚时分灯光逐渐亮起，您可以在此租借一套汉服，在亭台楼阁间留下美美的家庭合影。

**晚间：大唐不夜城（视觉盛宴）**
夜幕降临，直接从芙蓉园漫步至大唐不夜城。这是一条长达1.5公里的免费步行街。提醒您，这里的精髓在于**晚上19:00亮灯之后** [8]。届时，街区两侧的仿唐建筑灯火辉煌。沿途有不倒翁小姐姐的演艺（需提前很久排队）、壮观的贞观之治雕塑以及各种不重样的街头行为艺术。这里人流极大，请务必给孩子穿上颜色鲜艳的外套，或在手腕上系上防走失牵引带。
晚餐可在不夜城附近的“醉长安”解决，点一份正宗的油泼biangbiang面，感受热油浇在辣椒面上的刺啦声与浓郁香气。晚21:30左右，从大雁塔地铁站乘坐3号线换乘2号线返回酒店。

**Day 2 预算小结**
* 市内交通：约40元
* 景点门票：陕历博（按珍宝馆或讲解套票计）约150元 + 大雁塔约140元 + 芙蓉园240元
* 餐饮：午餐250元 + 晚餐150元
* **当日合计：约970元**

---

### Day 3：世界奇迹 —— 探访地下军团，纵享非遗美食

**上午：秦始皇兵马俑博物馆**
兵马俑距离市区约40公里。为了保障孩子有充足的睡眠，建议早上8:30从酒店出发。**最推荐的交通方案是“地铁+打车”**：先乘坐地铁到达9号线“华清池”站（C出口），出站后直接打车前往兵马俑，打车费仅约15元，全程耗时约75-90分钟，性价比极高且避免了旅游大巴的繁琐 [10]。需注意，连霍高速兵马俑专用线目前处于改建施工期，若您选择包车或自驾，需提前规划好绕行路线 [11]。
兵马俑门票实行全网实名制分时预约，成人120元，16周岁以下中国籍未成年人凭身份证免费（但也需在线预约免费票） [11]。景区内包含一、二、三号坑。一号坑最为震撼，但人流量极大。建议您在第一道检票口内的讲解服务处请一位官方讲解员（1-10人收费约100元，约1.5小时），让孩子了解“发丝不同”、“鞋底防滑纹”等军阶密码与历史细节 [11]。整个核心区域游览约需3.5小时。

**中午：永兴坊非遗美食体验**
约12:30出馆返回市区。如果您选择打车或包车返程，可让司机直接开往位于城墙中山门内的**永兴坊**。这里以陕西非遗美食为主，环境比传统的回民街更整洁有序。
午餐在这里边吃边逛：体验一把著名的“摔碗酒”（喝完米酒用力摔碎陶碗，寓意岁岁平安，孩子会觉得非常新奇好玩），再来一份子长煎饼或紫阳蒸盆子。这里的食物口味相对丰富，大人可以品尝水盆羊肉，小孩则可以吃香甜软糯的镜糕。一家三口午餐花费约150元。

**下午：满载而归，踏上返程**
在永兴坊稍作休息后，下午2:30左右取出行李。如果您的返程高铁在西安北站，建议乘坐地铁1号线换乘2号线或4号线直达北站，预留1.5小时安检时间。如果是飞机，建议提前预约送机专车前往咸阳机场。

**Day 3 预算小结**
* 交通：兵马俑往返（地铁+打车）约100元
* 景点门票：兵马俑240元 + 讲解100元
* 餐饮：午餐（永兴坊）150元
* **当日合计：约590元**

---

## 六、 核心景点门票与预约全攻略

西安作为顶流旅游城市，门票政策极其严格，以下为您梳理的关键信息请务必提前执行：

| 景点名称 | 票价政策 (成人) | 儿童/学生政策 | 预约/购票方式及关键时间点 |
| :--- | :--- | :--- | :--- |
| **秦始皇兵马俑** | 120元/人 | 16周岁(含)以下凭身份证免费；学生半价 | 公众号【秦始皇帝陵博物院】，提前7天放票。必须实名刷身份证入园 [11] |
| **陕西历史博物馆** | 免费 (珍宝馆30元) | 6岁以下/1.4米以下免费 | 公众号【陕西历史博物馆】，提前5天17:00放票。一票难求，建议购买珍宝馆票或第三方讲解套票 [6][7] |
| **西安明城墙** | 54元/人 | 1.2米以上27元/人 | 公众号【西安城墙】或现场购票，随买随用，日票当天不限次出入 [5] |
| **大唐芙蓉园** | 120元/人 | 1.2米以下免费；超过买半价票(60元) | 公众号【大唐芙蓉园】或【大唐芙蓉园景区】小程序，提前2-3天购票 [9] |
| **大雁塔(大慈恩寺)**| 40元/人，登塔25元 | 学生/儿童票半价 | 公众号【大慈恩寺】或现场购票 [8] |

---

## 七、 整体预算明细表与财务平衡

根据本路书的安排，一家三口（2大1小）3天2夜的总预算测算如下。基于您1.5万元的总预算，该方案留有极大的升级空间（如全程打车、升级高星酒店或增加演艺演出）：

| 支出类别 | 明细预算 (3人总计) | 备注说明 |
| :--- | :--- | :--- |
| **大交通 (往返)** | 约 3,000 - 6,000 元 | 视高铁或飞机、出发地远近而定 |
| **住宿 (2晚)** | 约 1,000 - 2,000 元 | 按每晚500-1000元的优质舒适型/中高档酒店计算 |
| **餐饮美食** | 约 1,000 - 1,500 元 | 每日100-200元/人的丰富地道美食体验 |
| **门票与讲解** | 约 800 - 1,200 元 | 含兵马俑、城墙、芙蓉园等核心大门票及专业讲解 |
| **市内与城际交通**| 约 400 - 800 元 | 包含地铁、打车及兵马俑往返交通 |
| **伴手礼/备用金** | 约 1,000 元 | 购买兵马俑模型、特产及应对突发情况 |
| **合计预估** | **约 7,200 - 12,500 元** | **轻松控制在1.5万元预算范围内，可按需升级** |

*如果您希望进一步消耗预算提升体验，强烈建议将Day 2晚上的大唐不夜城替换或增加一场沉浸式演出，例如前往华清宫观看大型实景历史舞剧《长恨歌》（票价298-988元不等），或带孩子前往浐灞观看有真骆驼和狼参与的《驼铃传奇》秀，不仅孩子大开眼界，也能极大地丰富文化体验 [12]。*

---

## 八、 亲子出行与深度游览避坑指南

**气候与穿搭提醒**：
若夏季（如目前8月）前往，西安素有“火炉”之称，气温极高且紫外线强。请务必为孩子准备透气吸汗的速干衣、遮阳帽和防晒霜。城墙和兵马俑部分区域遮挡较少，务必随身携带充足的饮用水（景区内水价通常翻倍）。冬季则干冷刺骨，需备好羽绒服和防风口罩。全行程请务必穿舒适防滑的运动鞋，城墙上的青砖路对高跟鞋和洞洞鞋极不友好。

**防坑与防骗提示**：
1. **拒绝野导与假景点**：前往兵马俑的路上，会有很多热情的“当地人”搭讪，说“5元送你到兵马俑”或推销“一日游”，这些100%是前往伪造展馆或蓝田玉购物店的陷阱。请认准官方博物馆全名：“秦始皇帝陵博物院” [11]。
2. **陕历博的黄牛陷阱**：在陕历博门口遇到有人说“50元带你进馆”切勿轻信，他们通常是通过非法手段刷您的身份信息，一旦被发现将被拉黑。建议直接买30元的珍宝馆门票入馆 [6]。
3. **真假游5路（306路）**：如果您选择从火车站乘坐游5路去兵马俑，请认准“白底红字”的正规大巴车，穿制服的工作人员在车下售票。任何灰色中巴或拉客喊“马上发车”的都是假车 [12]。

**亲子安全与节奏把控**：
本行程已为您留足了缓冲时间。每日行程基本保证中午可以回酒店休息或在咖啡店避暑。在人群密集的大唐不夜城和回民街，建议在孩子的口袋里放一张写有父母电话的小卡片。如果孩子在博物馆感到疲倦，不要为了“看完”而强行拖拽，适时在文创区购买一个吸引人的小礼品（如仕女盲盒或兵马俑手办）往往能重新激发他们的探索欲。

---

### 来源列表

[1] 中国铁路12306官方网站：https://www.12306.cn/
[2] 西安地铁线路及运营时间 - 携程旅行：https://my.trip.com/guide/transport/xian-subway.html
[3] 西安咸阳国际机场交通指南 - 携程旅行：https://flights.ctrip.com/booking/airport-xiy/jichangjiaotong.html
[4] 大众点评网：https://www.dianping.com/
[5] 西安城墙景区开放时间及门票（同程旅行）：https://m.ly.com/scenery/scenerydetail_7366.html
[6] 陕西历史博物馆门票预约最新规则 - 新浪新闻：https://k.sina.cn/article_7879776360_1d5abd86801901i48u.html
[7] 陕西历史博物馆门票预订（去哪儿网）：https://touch.piao.qunar.com/touch/detail.htm?id=22690
[8] 西安大雁塔及大唐不夜城游览攻略：https://k.sina.cn/article_7879776360_1d5abd86801901i48u.html
[9] 2026西安各景区五一门票预约时间（大唐芙蓉园）- 西安本地宝：https://m.xa.bendibao.com/jieri/72873.shtm
[10] 西安往返兵马俑最新交通方式（地铁9号线+打车）- 西安本地宝：http://xa.bendibao.com/tour/2024430/128180.shtm
[11] 秦始皇兵马俑博物馆2026预约及票价攻略 - GO CHINA：https://gochina.com.tw/first-time-xian-terracotta-army-guide
[12] 2026西安旅游超全攻略（演出、避坑指南）- Lynx猞猁旅行：https://onitrip.com/2026-xi-an-tourism-guide

In [6]:
from pathlib import Path
from travel_planner.memory import memory_store

path = Path("results/output_itinerary_sample_1.md")
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(result["final_itinerary"], encoding="utf-8")
memory_store.save_turn(
    "notebook-demo-user",
    skill_results={"xian_family_3d": {"path": str(path), "description": "西安亲子三日路书"}},
)
skill = memory_store.get_skill("notebook-demo-user", "xian_family_3d")
assert path.exists() and skill and skill["path"] == str(path)
print(f"路书已保存并注册为 Skill: {path}")

路书已保存并注册为 Skill: results\output_itinerary_sample_1.md


## MCP 服务调用示例

通过 MCP 协议连接旅游地图服务端，调用高德地图 API 获取天气/POI/路径规划等旅游情报。

In [7]:
from travel_planner.mcp.mcp_client import TravelMCPClient

async def mcp_demo():
    """连接 MCP 服务端并调用旅游工具"""
    async with TravelMCPClient() as client:
        # 1. 列出所有可用工具
        tools = await client.list_tools()
        print(f"共 {len(tools)} 个 MCP 工具可用")
        assert len(tools) == 10, f"MCP 工具数量异常: {len(tools)}"
        for t in tools:
            print(f"  - {t['name']}: {t['description'][:50]}")

        # 2. 查询目的地天气
        print("\n===== 北京 7 月天气 =====")
        weather = await client.call_tool("maps_weather", {"city": "北京"})
        assert '"error"' not in weather.lower(), weather
        print(weather)

        # 3. 搜索景点 POI
        print("\n===== 故宫博物院 POI =====")
        pois = await client.call_tool(
            "maps_text_search",
            {"keywords": "故宫博物院", "city": "北京", "top_k": 3}
        )
        assert '"error"' not in pois.lower(), pois
        print(pois)

        # 4. 周边搜索（故宫附近的餐厅）
        print("\n===== 故宫附近餐厅 =====")
        around = await client.call_tool(
            "maps_around_search",
            {"location": "116.397428,39.90923", "radius": "1000", "keywords": "餐厅", "top_k": 3}
        )
        assert '"error"' not in around.lower(), around
        print(around)

await mcp_demo()
print("Notebook 全链路执行成功")

[MCP] 已连接服务端，可用工具: ['maps_regeocode', 'maps_geo', 'maps_weather', 'maps_text_search', 'maps_around_search', 'maps_search_detail', 'maps_direction_transit_integrated', 'maps_direction_driving', 'maps_direction_walking', 'maps_distance']
共 10 个 MCP 工具可用
  - maps_regeocode: 将一个高德经纬度坐标（格式: 经度,纬度）转换为行政区划地址信息。

    旅游场景：用户给出景点
  - maps_geo: 将详细的结构化地址转换为经纬度坐标。支持对地标性名胜景区、建筑物名称解析为经纬度坐标。

    旅
  - maps_weather: 根据城市名称或 adcode 查询指定城市的天气预报。

    旅游场景：出行前查询目的地天气，用
  - maps_text_search: 关键词搜索 POI（景点、餐厅、酒店、车站等），返回相关信息。

    旅游场景：按关键词搜索目的
  - maps_around_search: 周边搜索：根据坐标和关键词，搜索指定半径内的 POI。

    旅游场景：在某个景点附近搜索餐厅/
  - maps_search_detail: 查询 POI ID 的详细信息（关键词搜索/周边搜索返回的 id）。

    旅游场景：拿到景点 
  - maps_direction_transit_integrated: 综合公共交通路径规划（火车/公交/地铁），跨城场景必须传起点与终点城市。

    旅游场景：规划跨
  - maps_direction_driving: 驾车路径规划，返回距离、时长与导航步骤。

    旅游场景：自驾游路书的交通方案规划。
    
  - maps_direction_walking: 步行路径规划（100km 以内），返回距离、时长与导航步骤。

    旅游场景：景点间步行距离估算
  - maps_distance: 测量多个起点到一个终点的距离，支持驾车/步行/球面距离。

    旅游场景：批量估算多个备选酒店到

==


[MCP] 调用工具 maps_text_search，参数 {'keywords': '故宫博物院', 'city': '北京', 'top_k': 3}
{
  "pois": [
    {
      "id": "B000A8UIN8",
      "name": "故宫博物院",
      "address": "景山前街4号",
      "tel": "4009501925",
      "type": "风景名胜;风景名胜;世界遗产|科教文化服务;博物馆;博物馆",
      "location": "116.397029,39.917839",
      "typecode": "110201|140100"
    },
    {
      "id": "B0FFKL520U",
      "name": "故宫博物院检票处",
      "address": "景山前街4号故宫博物院内(南侧)",
      "tel": [],
      "type": "生活服务;生活服务场所;生活服务场所",
      "location": "116.396952,39.913619",
      "typecode": "070000"
    },
    {
      "id": "B0FFFT7UKC",
      "name": "故宫博物院-文华殿",
      "address": "景山前街4号故宫博物院内(东南角)",
      "tel": "010-85007422",
      "type": "风景名胜;风景名胜;风景名胜",
      "location": "116.399345,39.915573",
      "typecode": "110200"
    }
  ]
}

===== 故宫附近餐厅 =====

[MCP] 调用工具 maps_around_search，参数 {'location': '116.397428,39.90923', 'radius': '1000', 'keywords': '餐厅', 'top_k': 3}
{
  "pois": [
    {
      "id": "B0FFG9V1R9",
      "name": "四季民福烤

Notebook 全链路执行成功
